In [1]:
import os
import ssl
import requests
import urllib3
from transformers import pipeline

# 1. Handle the SSL/Firewall block
ssl._create_default_https_context = ssl._create_unverified_context
os.environ['CURL_CA_BUNDLE'] = ''
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# 2. Simplified Pipeline Call
# We removed model_kwargs to avoid the "multiple values" TypeError
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

print("Downloading model...")
classifier = pipeline(
    "sentiment-analysis", 
    model=model_name
)

# 3. Test
print(classifier("The app is not bad"))

c:\Users\Almazt\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device set to use cpu


[{'label': 'POSITIVE', 'score': 0.9985883831977844}]


In [2]:
import os
import ssl
from transformers import AutoTokenizer, pipeline

# ==========================================
# 1. SSL & PROXY WORKAROUNDS (Corporate Environment)
# ==========================================

# Tell Python's built-in SSL module to ignore self-signed certs
ssl._create_default_https_context = ssl._create_unverified_context

# The golden environment variables to disable SSL verification globally for HTTP clients
os.environ['CURL_CA_BUNDLE'] = ''
os.environ['PYTHONHTTPSVERIFY'] = '0'
# Tell huggingface_hub specifically to skip verification if needed
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1' 

# ==========================================
# 2. INITIALIZE PIPELINE & TOKENIZER
# ==========================================
model_name = "distilbert-base-uncased-finetuned-sst-2-english"

try:
    print("Attempting to download/load model and tokenizer safely...")
    
    # Load tokenizer explicitly first
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Pass the explicit tokenizer into the pipeline to optimize memory usage
    classifier = pipeline("sentiment-analysis", model=model_name, tokenizer=tokenizer)
    print("✅ Model, tokenizer, and pipeline loaded successfully.\n")
    
except Exception as e:
    print(f"❌ Initialization failed. Error details: {e}")
    raise e

# ==========================================
# 3. HELPER FUNCTIONS & INFERENCE
# ==========================================

def smart_truncate(text):
    """Tokenize and truncate text to HuggingFace's 512 token max length limit, 
    then convert it back to a clean string."""
    tokens = tokenizer(text, truncation=True, max_length=512, add_special_tokens=True)
    return tokenizer.decode(tokens['input_ids'], skip_special_tokens=True)

def analyze_review(review):
    """Truncates text safely and routes it to the sentiment classifier."""
    truncated_review = smart_truncate(review)
    result = classifier(truncated_review)
    return result[0]

# --- Run Analysis ---
reviews = [
    "The app is not bad", 
    "The app is bad",
    "This is the most incredible experience I have ever had using a mobile interface!"
]

print("--- Running Individual Truncated Inference ---")
for r in reviews:
    analysis = analyze_review(r)
    print(f"Review: '{r}'")
    print(f"Label: {analysis['label']}, Score: {analysis['score']:.4f}\n")

print("--- Running Pipeline Native Batch Truncation ---")
batch_results = classifier(reviews, truncation=True, max_length=512)
for r, res in zip(reviews, batch_results):
    print(f"Review: '{r}' -> {res['label']} ({res['score']:.4f})")

Attempting to download/load model and tokenizer safely...


Device set to use cpu


✅ Model, tokenizer, and pipeline loaded successfully.

--- Running Individual Truncated Inference ---
Review: 'The app is not bad'
Label: POSITIVE, Score: 0.9986

Review: 'The app is bad'
Label: NEGATIVE, Score: 0.9998

Review: 'This is the most incredible experience I have ever had using a mobile interface!'
Label: POSITIVE, Score: 0.9998

--- Running Pipeline Native Batch Truncation ---
Review: 'The app is not bad' -> POSITIVE (0.9986)
Review: 'The app is bad' -> NEGATIVE (0.9998)
Review: 'This is the most incredible experience I have ever had using a mobile interface!' -> POSITIVE (0.9998)
